Image-Text Matching and Zero-Shot (Multi-label??) Classification Using CLIP. Considering MSCOCO as the support set

# Loading the CLIP model

# 2. Prepare the Inputs

In [18]:
import torch
import clip
from PIL import Image
import requests
import numpy as np
from pycocotools.coco import COCO
import os
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

IMAGE_DIR = '/home/BTECH_7TH_SEM/Documents/MML/MS-COCO/val2017/'
ANNOTATION_FILE = '/home/BTECH_7TH_SEM/Documents/MML/MS-COCO/annotations_trainval2017/annotations/instances_val2017.json'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# The 80 official MS-COCO class categories
# This is our "support set" of labels for zero-shot classification
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train',
    'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
    'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork',
    'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
    'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv',
    'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
    'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]
# https://docs.ultralytics.com/datasets/detect/coco/#dataset-yaml

print(f'{len(COCO_CLASSES) =}')


# 1. Load Model and Prepare Text Prompts
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
print(f"\nLoaded CLIP model on {DEVICE}")

# These text prompts are encoded only once for efficiency
text_inputs = clip.tokenize(COCO_CLASSES).to(DEVICE)
print(f'\n{text_inputs =}')
with torch.no_grad():
    text_features = model.encode_text(text_inputs)
    text_features /= text_features.norm(dim=-1, keepdim=True)

len(COCO_CLASSES) =80

Loaded CLIP model on cuda

text_inputs =tensor([[49406,  2533, 49407,  ...,     0,     0,     0],
        [49406, 11652, 49407,  ...,     0,     0,     0],
        [49406,  1615, 49407,  ...,     0,     0,     0],
        ...,
        [49406, 11798,  4298,  ...,     0,     0,     0],
        [49406,  2225, 39877,  ...,     0,     0,     0],
        [49406, 36841, 49407,  ...,     0,     0,     0]], device='cuda:0',
       dtype=torch.int32)


In [26]:
# --- 2. Evaluation Function ---
def evaluate_zero_shot_coco(num_test_images, coco_api, image_ids, threshold=24.5):
    """
    Evaluates CLIP's zero-shot performance on a subset of the COCO dataset.
    """
    print(f"\n--- Starting evaluation for {num_test_images} images ---")
    
    # Randomly select a subset of images for evaluation
    eval_image_ids = np.random.choice(image_ids, num_test_images, replace=False)
    
    all_predictions = []
    all_ground_truths = []

    for img_id in tqdm(eval_image_ids, desc="Evaluating images"):
        # --- Get Ground Truth Labels (Robust Method) ---
        # 1. Get all annotation IDs for the given image. Pass img_id as a list.
        ann_ids = coco_api.getAnnIds(imgIds=[img_id])
        # 2. Load all annotation metadata.
        anns = coco_api.loadAnns(ann_ids)
        # 3. Get all category names from the annotations.
        gt_categories = set()
        for ann in anns:
            # Ensure the annotation has a category ID
            if 'category_id' in ann:
                cat_id = ann['category_id']
                # Load the category info. loadCats expects a list.
                cats = coco_api.loadCats([cat_id])
                # Ensure the category was found and has a name
                if cats and 'name' in cats[0]:
                    gt_categories.add(cats[0]['name'])
        all_ground_truths.append(list(gt_categories))
        
        # --- Get CLIP Predictions ---
        img_info = coco_api.loadImgs([img_id])[0] # Pass img_id as a list
        img_path = os.path.join(IMAGE_DIR, img_info['file_name'])
        
        try:
            image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                image_features = model.encode_image(image)
                image_features /= image_features.norm(dim=-1, keepdim=True)
                
                similarity = (image_features @ text_features.T) * model.logit_scale.exp()
                
                pred_indices = torch.where(similarity[0] > threshold)[0].cpu().numpy()
                predicted_labels = [COCO_CLASSES[i] for i in pred_indices]
                all_predictions.append(predicted_labels)

        except (IOError, FileNotFoundError):
            print(f"Warning: Could not load image {img_path}. Skipping.")
            all_predictions.append([])

    # --- Calculate Metrics ---
    mlb = MultiLabelBinarizer(classes=COCO_CLASSES)
    gt_binarized = mlb.fit_transform(all_ground_truths)
    pred_binarized = mlb.transform(all_predictions)

    precision = precision_score(gt_binarized, pred_binarized, average='micro', zero_division=0)
    recall = recall_score(gt_binarized, pred_binarized, average='micro', zero_division=0)
    f1 = f1_score(gt_binarized, pred_binarized, average='micro', zero_division=0)
    
    print("\n--- Evaluation Results ---")
    print(f"Images Evaluated: {num_test_images}")
    print(f"Score Threshold: {threshold}")
    print(f"Micro-Precision: {precision:.4f}")
    print(f"Micro-Recall: {recall:.4f}")
    print(f"Micro-F1 Score: {f1:.4f}")
    print("--------------------------")


# --- Main Execution ---
# if __name__ == "__main__":
#     if not os.path.exists(ANNOTATION_FILE) or not os.path.exists(IMAGE_DIR):
#         print("Error: MS-COCO data not found. Please check your paths in the configuration.")
#     else:
#         # Load COCO API
#         coco = COCO(ANNOTATION_FILE)
#         all_image_ids = list(coco.imgs.keys())

#         # Run evaluations for the requested "testing" set sizes
#         # The "training" split size is irrelevant for zero-shot.
#         evaluate_zero_shot_coco(num_test_images=3000, coco_api=coco, image_ids=all_image_ids)
#         evaluate_zero_shot_coco(num_test_images=4000, coco_api=coco, image_ids=all_image_ids)
#         evaluate_zero_shot_coco(num_test_images=1000, coco_api=coco, image_ids=all_image_ids)


# --- Main Execution ---
if __name__ == "__main__":
    if not os.path.exists(ANNOTATION_FILE) or not os.path.exists(IMAGE_DIR):
        print("Error: MS-COCO data not found. Please check your ANNOTATION_FILE and IMAGE_DIR paths.")
    else:
        # 1. Load the COCO API
        coco = COCO(ANNOTATION_FILE)
        
        # 2. Get all image IDs from the validation set (which has 5000 images)
        all_image_ids = list(coco.imgs.keys())
        
        # 3. Shuffle the IDs for a random split
        np.random.seed(42) # Use a seed for reproducible results
        np.random.shuffle(all_image_ids)
        
        # 4. Define the split
        num_train_images = 2000
        train_image_ids = all_image_ids[:num_train_images]
        test_image_ids = all_image_ids[num_train_images:]
        
        print(f"Dataset split successfully.")
        print(f"Total images: {len(all_image_ids)}")
        print(f"Training images (held out): {len(train_image_ids)}")
        print(f"Testing images (for evaluation): {len(test_image_ids)}")
        
        # 5. Evaluate the model ONLY on the 3000 test images
        evaluate_zero_shot_coco(
            num_test_images=len(test_image_ids), 
            coco_api=coco, 
            image_ids=test_image_ids
        )

loading annotations into memory...
Done (t=0.16s)
creating index...
index created!
Dataset split successfully.
Total images: 5000
Training images (held out): 2000
Testing images (for evaluation): 3000

--- Starting evaluation for 3000 images ---


Evaluating images: 100%|██████████| 3000/3000 [00:19<00:00, 154.62it/s]



--- Evaluation Results ---
Images Evaluated: 3000
Score Threshold: 24.5
Micro-Precision: 0.6378
Micro-Recall: 0.2509
Micro-F1 Score: 0.3601
--------------------------


In [25]:
# --- 2. Evaluation Function ---
def evaluate_and_show_sample(img_id, coco_api, threshold=24.5):
    """
    Loads a single image, evaluates it, and prints detailed results.
    """
    print(f"\n--- Detailed Analysis for Image ID: {img_id} ---")

    # --- 1. Load Image and Get Ground Truth ---
    img_info = coco_api.loadImgs([img_id])[0]
    image_url = img_info.get('coco_url')
    
    try:
        image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")
        preprocessed_image = preprocess(image).unsqueeze(0).to(DEVICE)
    except (requests.exceptions.RequestException, IOError) as e:
        print(f"Error loading image: {e}")
        return

    ann_ids = coco_api.getAnnIds(imgIds=[img_id])
    anns = coco_api.loadAnns(ann_ids)
    ground_truth_labels = {coco_api.loadCats(ann['category_id'])[0]['name'] for ann in anns}

    # --- 2. Get CLIP Predictions and Probabilities ---
    with torch.no_grad():
        image_features = model.encode_image(preprocessed_image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        
        # Calculate similarity scores (logits)
        logits = (image_features @ text_features.T) * model.logit_scale.exp()
        
        # Get probabilities by applying sigmoid to logits
        probs = torch.sigmoid(logits[0])
        
        # Get predicted labels by applying a threshold to the logits
        pred_indices = torch.where(logits[0] > threshold)[0].cpu().numpy()
        predicted_labels = {COCO_CLASSES[i] for i in pred_indices}

    # --- 3. Display Results ---
    print(f"Image URL: {image_url}")
    print(f"\n✅ Ground-Truth Labels: {sorted(list(ground_truth_labels))}")
    print(f"🤖 Predicted Labels (Threshold > {threshold}): {sorted(list(predicted_labels))}")
    
    # Create a sorted list of all class probabilities
    all_class_probs = sorted(zip(COCO_CLASSES, probs.cpu().numpy()), key=lambda x: x[1], reverse=True)
    
    print("\n--- Caption Probabilities (Top 10) ---")
    for label, prob in all_class_probs[:10]:
        print(f"  - {label:<20} | Probability: {prob:.4f}")


    # --- Calculate Metrics ---
    mlb = MultiLabelBinarizer(classes=COCO_CLASSES)
    gt_binarized = mlb.fit_transform(all_ground_truths)
    pred_binarized = mlb.transform(all_predictions)

    precision = precision_score(gt_binarized, pred_binarized, average='micro', zero_division=0)
    recall = recall_score(gt_binarized, pred_binarized, average='micro', zero_division=0)
    f1 = f1_score(gt_binarized, pred_binarized, average='micro', zero_division=0)
    
    print("\n--- Evaluation Results ---")
    print(f"Images Evaluated: {num_test_images}")
    print(f"Score Threshold: {threshold}")
    print(f"Micro-Precision: {precision:.4f}")
    print(f"Micro-Recall: {recall:.4f}")
    print(f"Micro-F1 Score: {f1:.4f}")
    print("--------------------------")


# --- Main Execution ---
if __name__ == "__main__":
    if not os.path.exists(ANNOTATION_FILE) or not os.path.exists(IMAGE_DIR):
        print("Error: MS-COCO data not found. Please check your paths in the configuration.")
    else:
        # Load COCO API
        coco = COCO(ANNOTATION_FILE)
        all_image_ids = list(coco.imgs.keys())

        # Run evaluations for the requested "testing" set sizes
        # The "training" split size is irrelevant for zero-shot.
        evaluate_zero_shot_coco(num_test_images=3000, coco_api=coco, image_ids=all_image_ids)
        evaluate_zero_shot_coco(num_test_images=4000, coco_api=coco, image_ids=all_image_ids)
        evaluate_zero_shot_coco(num_test_images=1000, coco_api=coco, image_ids=all_image_ids)

        test_image_id = 397133 
#
#     # Run the detailed analysis
        evaluate_and_show_sample(test_image_id, coco_api=coco)

loading annotations into memory...
Done (t=0.16s)
creating index...
index created!

--- Starting evaluation for 3000 images ---


Evaluating images: 100%|██████████| 3000/3000 [00:16<00:00, 181.76it/s]



--- Evaluation Results ---
Images Evaluated: 3000
Score Threshold: 24.5
Micro-Precision: 0.6448
Micro-Recall: 0.2535
Micro-F1 Score: 0.3639
--------------------------

--- Starting evaluation for 4000 images ---


Evaluating images: 100%|██████████| 4000/4000 [00:21<00:00, 183.97it/s]



--- Evaluation Results ---
Images Evaluated: 4000
Score Threshold: 24.5
Micro-Precision: 0.6433
Micro-Recall: 0.2532
Micro-F1 Score: 0.3634
--------------------------

--- Starting evaluation for 1000 images ---


Evaluating images: 100%|██████████| 1000/1000 [00:05<00:00, 182.97it/s]



--- Evaluation Results ---
Images Evaluated: 1000
Score Threshold: 24.5
Micro-Precision: 0.6221
Micro-Recall: 0.2571
Micro-F1 Score: 0.3639
--------------------------

--- Detailed Analysis for Image ID: 397133 ---
Image URL: http://images.cocodataset.org/val2017/000000397133.jpg

✅ Ground-Truth Labels: ['bottle', 'bowl', 'broccoli', 'carrot', 'cup', 'dining table', 'knife', 'oven', 'person', 'sink', 'spoon']
🤖 Predicted Labels (Threshold > 24.5): []

--- Caption Probabilities (Top 10) ---
  - person               | Probability: 1.0000
  - bicycle              | Probability: 1.0000
  - car                  | Probability: 1.0000
  - motorcycle           | Probability: 1.0000
  - airplane             | Probability: 1.0000
  - bus                  | Probability: 1.0000
  - train                | Probability: 1.0000
  - truck                | Probability: 1.0000
  - boat                 | Probability: 1.0000
  - traffic light        | Probability: 1.0000


NameError: name 'all_ground_truths' is not defined